# Creating an ALttPR Dataset

This will create a file `alttpr.csv` assuming you have done the following:

* (Legally) acquire an unheadered Japanese v1.0 copy of A Link to the Past
* Set up and install the PHP Z3 randomizer https://github.com/sporchia/alttp_vt_randomizer
* Acquire a Mudora binary https://github.com/alttpr-mudora/mudora

Use these to generate randomized ALttPR ROMs for inspection by Mudora:

```php
php artisan alttp:randomize <ROM> <output_dir> \
  --goal=ganon \
  --state=open \
  --weapons=randomized \
  --glitches=none \
  --crystals_ganon=7 \
  --crystals_tower=7 \
  --item_placement=advanced \
  --dungeon_items=standard \
  --accessibility=items \
  --hints=on \
  --item_pool=normal \
  --item_functionality=normal
```

Verify you can use Mudora to inspect the ROM:

```sh
./mudora.exe -v
./mudora.exe -r <output_dir>/<ROM>
```

Also set up your `.env` file with a path to the output ROMs:

```sh
cp .env.example .env
```

In [1]:
from pathlib import Path
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import re
from collections import defaultdict
import dotenv
import os

import pandas as pd

dotenv.load_dotenv(dotenv_path='.env')

ROMS_PATH = Path(os.environ['ROM_PATH'])
GAMES_PATH = Path("games.json")
CSV_PATH = Path("alttpr.csv")

## Count the number of ROMs for processing

In [2]:
files = list(ROMS_PATH.glob("*.sfc"))

print(f"File count: {len(files):,}")

File count: 28,193


## Set up batching 

In [3]:
BATCH_SZ = 500

batches = [files[i:i + BATCH_SZ] for i in range(0, len(files), BATCH_SZ)]

print(f"Batch count: {len(batches):,}")

Batch count: 57


## Process each batch with Mudora using 5 worker threads

In [4]:
def process_batch(batch):
    results = []

    for f in batch:
        result = subprocess.run(["./mudora.exe", "-rom", f.resolve()], capture_output=True, text=True)
        results.append((f, result))

    return results


results = []
completed = 0
start = time.perf_counter()

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_batch, batch) for batch in batches]

    for future in as_completed(futures):
        results.extend(future.result())
        completed += 1

        elapsed = time.perf_counter() - start
        avg_batch_time = elapsed / completed
        est_remaining = avg_batch_time * (len(batches) - completed)

        print(f"Completed {completed}/{len(batches)} batches ({completed / len(batches) * 100:.2f}%)")
        print(f"\tAverage batch time: {avg_batch_time:.2f}s")
        print(f"\tEstimated remaining time: {est_remaining:.2f}s")


print(f"Processed {len(results):,} files")

Completed 1/57 batches (1.75%)
	Average batch time: 64.50s
	Estimated remaining time: 3611.83s
Completed 2/57 batches (3.51%)
	Average batch time: 32.51s
	Estimated remaining time: 1788.07s
Completed 3/57 batches (5.26%)
	Average batch time: 21.76s
	Estimated remaining time: 1174.88s
Completed 4/57 batches (7.02%)
	Average batch time: 16.35s
	Estimated remaining time: 866.61s
Completed 5/57 batches (8.77%)
	Average batch time: 13.14s
	Estimated remaining time: 683.41s
Completed 6/57 batches (10.53%)
	Average batch time: 21.24s
	Estimated remaining time: 1083.33s
Completed 7/57 batches (12.28%)
	Average batch time: 18.30s
	Estimated remaining time: 914.98s
Completed 8/57 batches (14.04%)
	Average batch time: 16.07s
	Estimated remaining time: 787.23s
Completed 9/57 batches (15.79%)
	Average batch time: 14.29s
	Estimated remaining time: 686.10s
Completed 10/57 batches (17.54%)
	Average batch time: 12.97s
	Estimated remaining time: 609.45s
Completed 11/57 batches (19.30%)
	Average batch ti

## Checkpoint with JSON output

This step creates an intermediate JSON file in case something goes wrong later. We can read the results from JSON instead of re-processing each ROM.

In [6]:
import json

games = defaultdict(list)

for i, (game_path, result) in enumerate(results):
    game_id = game_path.stem.split("_")[-1]

    games["GameId"].append(game_id)
    for line in result.stdout.split("\n"):
        
        if len(line) == 0 or line[0] != ' ':
            continue

        try:
            location, item = re.split(r"\s{2,}", line.strip())
        except:
            location = line.strip()
            item = "UNKNOWN"

        games[location].append(item)


    if i > 0 and i % 1000 == 0:
        print(f"Completed parsing {i} results")

with open(GAMES_PATH, "w") as fp:
    json.dump(games, fp)

Completed parsing 1000 results
Completed parsing 2000 results
Completed parsing 3000 results
Completed parsing 4000 results
Completed parsing 5000 results
Completed parsing 6000 results
Completed parsing 7000 results
Completed parsing 8000 results
Completed parsing 9000 results
Completed parsing 10000 results
Completed parsing 11000 results
Completed parsing 12000 results
Completed parsing 13000 results
Completed parsing 14000 results
Completed parsing 15000 results
Completed parsing 16000 results
Completed parsing 17000 results
Completed parsing 18000 results
Completed parsing 19000 results
Completed parsing 20000 results
Completed parsing 21000 results
Completed parsing 22000 results
Completed parsing 23000 results
Completed parsing 24000 results
Completed parsing 25000 results
Completed parsing 26000 results
Completed parsing 27000 results
Completed parsing 28000 results


## Load the intermediate representation into a dataframe

In [7]:
with open(GAMES_PATH, "r") as fp:
    data = json.load(fp)

df = pd.DataFrame(data)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28193 entries, 0 to 28192
Columns: 228 entries, GameId to Pyramid Bottle
dtypes: object(228)
memory usage: 49.0+ MB


,GameId,Sahasrahla's Hut - Left,Sahasrahla's Hut - Middle,Sahasrahla's Hut - Right,Sahasrahla,Zora's Ledge,Waterfall Fairy - Left,Waterfall Fairy - Right,Master Sword Pedestal,King's Tomb,...,Ganon's Tower - Compass Room - Bottom Right,Ganon's Tower - Big Key Chest,Ganon's Tower - Big Key Room - Left,Ganon's Tower - Big Key Room - Right,Ganon's Tower - Mini Helmasaur Room - Left,Ganon's Tower - Mini Helmasaur Room - Right,Ganon's Tower - Pre-Moldorm Chest,Ganon's Tower - Moldorm Chest,Waterfall Bottle,Pyramid Bottle
0,z232byRRkpnx3Ao,Piece of Heart,Boomerang,Progressive Armor,Piece of Heart,Progressive Shield,Heart Container,Piece of Heart,Arrows (10),Fire Rod,...,Key (Ganon's Tower),Bombs (3),Arrows (10),Rupee (Red),Heart Container,Rupees (50),Bombs (3),Compass (Ganon's Tower),Bottle (Blue Potion),Bottle (Green Potion)
1,1xXJNBQBNaPeXWP,Rupee (Red),Progressive Shield,Rupee (Red),Piece of Heart,Rupees (100),Piece of Heart,Bombs (3),Heart Container,Piece of Heart,...,Bombs (3),Heart Container,Book of Mudora,Piece of Heart,Piece of Heart,Rupees (50),Rupee (Red),Rupee (Red),Bottle (Red Potion),Bottle (Green Potion)
2,mOX4bAYYnLowq0v,Rupees (300),Cane of Somaria,Arrows (10),Rupee (Red),Bottle (Faerie),Rupees (300),Bombs (3),Bottle (Green Potion),Progressive Sword,...,Rupee (Green),Rupee (Red),Big Key (Ganon's Tower),Arrows (10),Powder,Rupee (Red),Compass (Ganon's Tower),Rupees (50),Bottle (Green Potion),Bottle (Bee)
3,vPlbLrZrrL4K3a0,Piece of Heart,Bombs (3),Rupees (100),Rupee (Green),Rupees (50),Piece of Heart,Piece of Heart,Rupee (Red),Piece of Heart,...,Bombs (3),Big Key (Ganon's Tower),Piece of Heart,Bombs (3),Book of Mudora,Key (Ganon's Tower),Heart Container,Rupee (Red),Bottle (Bee),Bottle (Blue Potion)
4,JbqRx1Qd5jKpXm5,Rupees (50),Quake,Rupee (Red),Arrows (10),Piece of Heart,Rupees (300),Piece of Heart,Rupee (Red),Piece of Heart,...,Arrows (10),Piece of Heart,Heart Container,Rupee (Red),Piece of Heart,Progressive Shield,Piece of Heart,Rupee (Red),Bottle (Super bee),Bottle (Green Potion)


## Data cleaning

Mudora sometimes cannot read the ROM (for whatever reason). If that happens we should drop the row entirely. It still produces a result, just puts "UNKNOWN" where the value should be.

In [8]:
# Drop the rows with unkowns
df = df[~df.isin(["UNKNOWN"]).any(axis=1)].reset_index(drop=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28193 entries, 0 to 28192
Columns: 228 entries, GameId to Pyramid Bottle
dtypes: object(228)
memory usage: 49.0+ MB


## Write the dataframe to CSV

In [9]:
df.to_csv(CSV_PATH)